# Donor Upgrade Scoring
**Lighthouse Sanctuary — ML Pipeline (Lighter)**

Predict next gift amount (PHP) using OLS regression to identify donors with upgrade potential.

---
## 1. Problem Framing

### Business Problem
Identify donors most likely to give a **larger gift next time** — these are candidates for personal outreach, matching gift opportunities, or major donor cultivation. This is an **explanatory** regression model to understand which factors predict gift size growth.

### ML Task
- **Type:** Regression (OLS — explanatory)
- **Target:** `avg_gift_size` (PHP) — proxy for next gift amount
- **Unit:** One row per supporter
- **Goal:** Interpret coefficients to understand upgrade drivers, not maximize predictive R²

### Features
| Feature | Description |
|---|---|
| `total_lifetime_value` | Historical giving depth |
| `donation_frequency` | Giving cadence |
| `num_campaigns` | Breadth of engagement |
| `is_recurring_donor` | Signals commitment |
| `days_since_last_donation` | Recency |
| `acquisition_channel` | Source channel |
| `supporter_type` | Donor category |

### Model
OLS (statsmodels) — full coefficient table with p-values and confidence intervals

### Deployment
No API endpoint — insights inform manual outreach strategy and donor segmentation.

---
## 2. Data Acquisition, Preparation & Exploration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import subprocess, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.preprocessing import LabelEncoder

NB_DIR    = Path().resolve()
ML_DIR    = NB_DIR.parent if NB_DIR.name == 'notebooks' else NB_DIR / 'ml-pipelines'
DATA_DIR  = ML_DIR / 'data'

DONOR_MASTER = DATA_DIR / 'donor_master.csv'
if not DONOR_MASTER.exists():
    subprocess.run([sys.executable, str(ML_DIR / 'master_dataset_builder.py')], check=True)

df = pd.read_csv(DONOR_MASTER, low_memory=False)

# Filter to donors with at least one donation
df = df[df['total_donations'] > 0].copy()
print(f'Donors with at least one donation: {len(df)}')
df[['avg_gift_size', 'total_lifetime_value', 'donation_frequency', 'num_campaigns']].describe()

In [ ]:
# Distribution of avg_gift_size
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['avg_gift_size'].dropna(), bins=40, color='#3498db', edgecolor='white')
axes[0].set_title('Distribution of Avg Gift Size (PHP)')
axes[0].set_xlabel('PHP')

axes[1].hist(np.log1p(df['avg_gift_size'].dropna()), bins=40, color='#2ecc71', edgecolor='white')
axes[1].set_title('Log(1 + Avg Gift Size)')
axes[1].set_xlabel('log(PHP)')

plt.tight_layout()
plt.show()

# Gift size by supporter type
plt.figure(figsize=(10, 4))
if 'supporter_type' in df.columns:
    df.boxplot(column='avg_gift_size', by='supporter_type')
    plt.title('Avg Gift Size by Supporter Type')
    plt.suptitle('')
    plt.tight_layout()
    plt.show()

In [ ]:
# Scatter plots of key predictors vs gift size
predictors = ['total_lifetime_value', 'donation_frequency', 'num_campaigns', 'days_since_last_donation']
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, col in zip(axes, predictors):
    if col in df.columns:
        ax.scatter(df[col], df['avg_gift_size'], alpha=0.3, s=15, color='#3498db')
        ax.set_xlabel(col)
        ax.set_ylabel('Avg Gift Size')
        ax.set_title(f'{col} vs Gift Size')

plt.tight_layout()
plt.show()

---
## 3. Modeling & Feature Selection

In [ ]:
# Encode categoricals
model_df = df.copy()

for c in ['acquisition_channel', 'supporter_type']:
    if c in model_df.columns:
        model_df[c] = model_df[c].fillna('Unknown')
        le = LabelEncoder()
        model_df[c + '_enc'] = le.fit_transform(model_df[c])

TARGET = 'avg_gift_size'
FEATURES = [
    'total_lifetime_value', 'donation_frequency', 'num_campaigns',
    'is_recurring_donor', 'days_since_last_donation',
    'acquisition_channel_enc', 'supporter_type_enc'
]
FEATURES = [f for f in FEATURES if f in model_df.columns]

model_df = model_df[FEATURES + [TARGET]].dropna()
print(f'Model dataset: {model_df.shape}')

# Log-transform target (right-skewed)
model_df['log_gift_size'] = np.log1p(model_df[TARGET])

X = sm.add_constant(model_df[FEATURES].fillna(model_df[FEATURES].median()))
y = model_df['log_gift_size']

ols = sm.OLS(y, X).fit()
print(ols.summary())

---
## 4. Evaluation & Interpretation

In [ ]:
print(f'R²:          {ols.rsquared:.4f}')
print(f'Adj. R²:     {ols.rsquared_adj:.4f}')
print(f'F-stat:      {ols.fvalue:.2f} (p={ols.f_pvalue:.4f})')

# Coefficient plot
coef_df = pd.DataFrame({
    'Feature': ols.params.index,
    'Coef': ols.params.values,
    'CI_low': ols.conf_int()[0].values,
    'CI_high': ols.conf_int()[1].values,
    'p': ols.pvalues.values
})
coef_df = coef_df[coef_df['Feature'] != 'const'].sort_values('Coef')

plt.figure(figsize=(9, 5))
colors = ['#e74c3c' if p < 0.05 else '#95a5a6' for p in coef_df['p']]
plt.barh(coef_df['Feature'], coef_df['Coef'], color=colors,
         xerr=coef_df['CI_high'] - coef_df['Coef'])
plt.axvline(0, color='black', linewidth=0.8)
plt.title('OLS Coefficients (red = p<0.05)\nTarget: log(1 + Avg Gift Size)')
plt.tight_layout()
plt.show()

print('\nSignificant predictors of larger gift size:')
sig = coef_df[coef_df['p'] < 0.05].sort_values('Coef', ascending=False)
for _, row in sig.iterrows():
    direction = 'increases' if row['Coef'] > 0 else 'decreases'
    print(f'  {row["Feature"]:35s} {direction} gift size (coef={row["Coef"]:.4f}, p={row["p"]:.4f})')

In [ ]:
# Residual diagnostics
fitted = ols.fittedvalues
residuals = ols.resid

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(fitted, residuals, alpha=0.4, s=15, color='#3498db')
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Fitted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted')

from scipy import stats
stats.probplot(residuals, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot of Residuals')

plt.tight_layout()
plt.show()

---
## 5. Causal and Relationship Analysis

In [ ]:
# Upgrade opportunity segmentation
df_active = df[df['is_churned'] == 0].copy()
df_active['upgrade_score'] = (
    df_active['donation_frequency'].rank(pct=True) * 0.4 +
    df_active['num_campaigns'].rank(pct=True) * 0.3 +
    df_active['is_recurring_donor'].fillna(0) * 0.3
)

print('=== Top Upgrade Candidates ===')
top_upgrades = df_active.nlargest(10, 'upgrade_score')[
    ['display_name', 'supporter_type', 'avg_gift_size', 'donation_frequency',
     'num_campaigns', 'upgrade_score']
]
print(top_upgrades.to_string(index=False))

In [ ]:
# Interaction: does acquisition channel moderate the frequency→gift relationship?
if 'acquisition_channel' in df.columns:
    channel_stats = df.groupby('acquisition_channel').agg(
        avg_gift=('avg_gift_size', 'mean'),
        avg_freq=('donation_frequency', 'mean'),
        count=('supporter_id', 'count')
    ).sort_values('avg_gift', ascending=False)
    print('=== Gift Size and Frequency by Acquisition Channel ===')
    print(channel_stats.to_string())

---
## 6. Deployment Notes

This model is **explanatory only** — it is not deployed to an API endpoint.

### How to use the outputs
1. Export `upgrade_score` column from the segmentation cell above into the CRM/admin dashboard
2. Filter active donors with `upgrade_score > 0.7` for personal outreach
3. Share the coefficient plot with the fundraising team — it shows what matters for gift growth

### If deploying in the future
- Switch to a Random Forest or XGBoost regressor for better predictive accuracy
- Target variable could be changed to `next_gift_amount` if longitudinal data is collected
- Integrate with the .NET backend to surface upgrade scores in the donor profile view

### Limitations
- OLS assumes linearity — gift size may have non-linear relationships with features
- R² may be low if most variance in gift size is driven by donor-level idiosyncrasies
- Update quarterly with new donation data